SALES COMMERCIAL ANALYTICS PROJECT FOCUSING ON A B2B TECHNOLOGY COMPANY THAT SELLS SEVERAL TECHNOLOGY PRODUCTS TO BUSINESS CUSTOMERS. 

We are trying to answer how effecitvely a company is converting its sales pipeline into revenue, and what factors are associated with stronger or weaker sales performances



In [154]:
#Importing necessary libraries
import pandas as pd

df = pd.read_csv("../data/sales_pipeline.csv")
print("First 5 Rows:")

print(df.head(11))
print("\n Dataframe information:")
df.info()



First 5 Rows:
   opportunity_id      sales_agent         product     account deal_stage  \
0        1C1I7A6R      Moses Frase  GTX Plus Basic     Cancity        Won   
1        Z063OYW0  Darcel Schlecht          GTXPro       Isdom        Won   
2        EC4QE1BX  Darcel Schlecht      MG Special     Cancity        Won   
3        MV1LWRNH      Moses Frase       GTX Basic     Codehow        Won   
4        PE84CX4O        Zane Levy       GTX Basic      Hatfan        Won   
5        ZNBS69V1    Anna Snelling      MG Special    Ron-tech        Won   
6        9ME3374G   Vicki Laflamme      MG Special     J-Texon        Won   
7        7GN8Q4LL   Markita Hansen       GTX Basic      Cheers        Won   
8        OLK9LKZB  Niesha Huffines  GTX Plus Basic    Zumgoity        Won   
9        HAXMC4IX   James Ascencio     MG Advanced         NaN   Engaging   
10       NL3JZH1Z    Anna Snelling      MG Special  Bioholding        Won   

   engage_date  close_date  close_value  
0   2016-10-20  201

Investigating Pipeline Health

In [155]:
#Investigating the number of rows, the columns and opportunities 
df.info()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 8800 entries, 0 to 8799
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   opportunity_id  8800 non-null   str    
 1   sales_agent     8800 non-null   str    
 2   product         8800 non-null   str    
 3   account         7375 non-null   str    
 4   deal_stage      8800 non-null   str    
 5   engage_date     8300 non-null   str    
 6   close_date      6711 non-null   str    
 7   close_value     6711 non-null   float64
dtypes: float64(1), str(7)
memory usage: 1.1 MB


Index(['opportunity_id', 'sales_agent', 'product', 'account', 'deal_stage',
       'engage_date', 'close_date', 'close_value'],
      dtype='str')

In [156]:
df["deal_stage"].unique()

<ArrowStringArray>
['Won', 'Engaging', 'Lost', 'Prospecting']
Length: 4, dtype: str

In [157]:
df["deal_stage"].value_counts()

deal_stage
Won            4238
Lost           2473
Engaging       1589
Prospecting     500
Name: count, dtype: int64

In [158]:
df.isnull().sum()

opportunity_id       0
sales_agent          0
product              0
account           1425
deal_stage           0
engage_date        500
close_date        2089
close_value       2089
dtype: int64

In [159]:
df[df["deal_stage"]=="Engaging"]["close_date"].isnull().sum()

np.int64(1589)

In [160]:
df[df["deal_stage"]=="Prospecting"]["close_date"].isnull().sum()

np.int64(500)

In [161]:
df[df["deal_stage"]=="Won"]["close_date"].isnull().sum()

np.int64(0)

In [162]:
df[df["deal_stage"]=="Lost"]["close_date"].isnull().sum()

np.int64(0)

### Missing Value Investigation

The dataset contains 2,089 missing values in both `close_date` and
`close_value`. These records correspond exactly to the 1,589 Engaging
and 500 Prospecting opportunities.

All Won and Lost opportunities have a recorded `close_date`.

This suggests that the missing `close_date` and `close_value` values are
related to opportunities that have not yet reached a closed stage, rather
than necessarily representing missing or incorrect data.

In [163]:
df[df["account"].isnull()]["deal_stage"].value_counts()

deal_stage
Engaging       1088
Prospecting     337
Name: count, dtype: int64

### Account Data Quality

There are 1,425 opportunities with missing account information.
All of these records are in the Engaging or Prospecting stages.

This means the missing account information is concentrated entirely
within open opportunities. The reason for this pattern should be
investigated before deciding whether the missing values require
treatment.

In [164]:
df["close_value"].sum()

np.float64(10005534.0)

In [165]:
df[df["deal_stage"]=="Won"]["close_value"].sum()

np.float64(10005534.0)

In [166]:
df[df["deal_stage"]=="Lost"]["close_value"].sum()

np.float64(0.0)

In [167]:
df[df["deal_stage"]=="Won"]["close_value"].mean()

np.float64(2360.9093912222747)

In [168]:
value_count_won = df[df["deal_stage"]=="Won"]["opportunity_id"].count()
value_count_lost = df[df["deal_stage"]=="Lost"]["opportunity_id"].count()

percentage_for_value_count = value_count_won / (value_count_won + value_count_lost) * 100
print(percentage_for_value_count)

63.15005215318135


In [169]:
df[df["deal_stage"]=="Won"].groupby("product")["close_value"].sum()

product
GTK 500            400612.0
GTX Basic          499263.0
GTX Plus Basic     705275.0
GTX Plus Pro      2629651.0
GTXPro            3510578.0
MG Advanced       2216387.0
MG Special          43768.0
Name: close_value, dtype: float64

In [170]:
df[df["deal_stage"]=="Won"].groupby("product")["opportunity_id"].count()

product
GTK 500            15
GTX Basic         915
GTX Plus Basic    653
GTX Plus Pro      479
GTXPro            729
MG Advanced       654
MG Special        793
Name: opportunity_id, dtype: int64

In [171]:
df[df["deal_stage"]=="Won"].groupby("product")["close_value"].mean()

product
GTK 500           26707.466667
GTX Basic           545.642623
GTX Plus Basic     1080.053599
GTX Plus Pro       5489.876827
GTXPro             4815.607682
MG Advanced        3388.970948
MG Special           55.192938
Name: close_value, dtype: float64

In [172]:
df[(df["deal_stage"] == "Won") & (df["product"] == "GTK 500")]["close_value"]

59      25897.0
677     30288.0
1156    29617.0
1825    25288.0
2559    23746.0
2586    24949.0
2808    27385.0
3352    24745.0
3495    24899.0
3522    25791.0
3839    29166.0
3925    25464.0
6420    26186.0
7848    27971.0
8078    29220.0
Name: close_value, dtype: float64

In [173]:
df[df["product"] == "GTK 500"]["deal_stage"].value_counts()

deal_stage
Won         15
Engaging    15
Lost        10
Name: count, dtype: int64

In [174]:
Won_by_product = df[df["deal_stage"] == "Won"].groupby("product")["opportunity_id"].count()


In [175]:
lost_by_product = df[df["deal_stage"] == "Lost"].groupby("product")["opportunity_id"].count()

In [176]:
win_rate_by_product = Won_by_product/(Won_by_product+lost_by_product)*100
print(win_rate_by_product)

product
GTK 500           60.000000
GTX Basic         63.718663
GTX Plus Basic    62.131304
GTX Plus Pro      64.295302
GTXPro            63.557105
MG Advanced       60.332103
MG Special        64.840556
Name: opportunity_id, dtype: float64


In [177]:
df[df["deal_stage"] == "Won"].groupby("product")["close_value"].sum()

product
GTK 500            400612.0
GTX Basic          499263.0
GTX Plus Basic     705275.0
GTX Plus Pro      2629651.0
GTXPro            3510578.0
MG Advanced       2216387.0
MG Special          43768.0
Name: close_value, dtype: float64

In [178]:
won_value_by_product = df[df["deal_stage"] == "Won"].groupby("product")["close_value"].sum()
total_sum = df[df["deal_stage"]=="Won"]["close_value"].sum()
product_percentage_share = won_value_by_product/(total_sum)*100
print(product_percentage_share)

product
GTK 500            4.003904
GTX Basic          4.989869
GTX Plus Basic     7.048849
GTX Plus Pro      26.281966
GTXPro            35.086363
MG Advanced       22.151611
MG Special         0.437438
Name: close_value, dtype: float64


In [179]:
total_deal_stage_count = df["deal_stage"].value_counts()

engaging_deal = total_deal_stage_count["Engaging"]
prospecting_deal = total_deal_stage_count["Prospecting"]

deal_stage_unclosed = engaging_deal + prospecting_deal 
print(deal_stage_unclosed)

percetange_of_deal_stage_unclosed = deal_stage_unclosed/df["opportunity_id"].count()*100
print(percetange_of_deal_stage_unclosed)

2089
23.738636363636363


In [180]:
percentage_of_deals_engaged = engaging_deal/(deal_stage_unclosed)*100
print(percentage_of_deals_engaged)

76.06510292005744


In [181]:
df[df["deal_stage"]=="Won"].groupby("sales_agent")["opportunity_id"].count()

sales_agent
Anna Snelling         208
Boris Faz             101
Cassey Cress          163
Cecily Lampkin        107
Corliss Cosme         150
Daniell Hammack       114
Darcel Schlecht       349
Donn Cantrell         158
Elease Gluck           80
Garret Kinder          75
Gladys Colclough      135
Hayden Neloms         107
James Ascencio        135
Jonathan Berthelot    171
Kami Bicknell         174
Kary Hendrixson       209
Lajuana Vencill       127
Markita Hansen        130
Marty Freudenburg     122
Maureen Marcano       149
Moses Frase           129
Niesha Huffines       105
Reed Clapper          155
Rosalina Dieter        72
Rosie Papadopoulos     78
Versie Hillebrand     176
Vicki Laflamme        221
Violet Mclelland      122
Wilburn Farren         55
Zane Levy             161
Name: opportunity_id, dtype: int64

In [183]:
df[df["deal_stage"]=="Won"].groupby("sales_agent")["close_value"].sum()

sales_agent
Anna Snelling          275056.0
Boris Faz              261631.0
Cassey Cress           450489.0
Cecily Lampkin         229800.0
Corliss Cosme          421036.0
Daniell Hammack        364229.0
Darcel Schlecht       1153214.0
Donn Cantrell          445860.0
Elease Gluck           289195.0
Garret Kinder          197773.0
Gladys Colclough       345674.0
Hayden Neloms          272111.0
James Ascencio         413533.0
Jonathan Berthelot     284886.0
Kami Bicknell          316456.0
Kary Hendrixson        454298.0
Lajuana Vencill        194632.0
Markita Hansen         328792.0
Marty Freudenburg      291195.0
Maureen Marcano        350395.0
Moses Frase            207182.0
Niesha Huffines        176961.0
Reed Clapper           438336.0
Rosalina Dieter        235403.0
Rosie Papadopoulos     230169.0
Versie Hillebrand      187693.0
Vicki Laflamme         478396.0
Violet Mclelland       123431.0
Wilburn Farren         157640.0
Zane Levy              430068.0
Name: close_value, dtype: fl

In [184]:
df[df["deal_stage"]=="Won"].groupby("sales_agent")["close_value"].mean()

sales_agent
Anna Snelling         1322.384615
Boris Faz             2590.405941
Cassey Cress          2763.736196
Cecily Lampkin        2147.663551
Corliss Cosme         2806.906667
Daniell Hammack       3194.991228
Darcel Schlecht       3304.338109
Donn Cantrell         2821.898734
Elease Gluck          3614.937500
Garret Kinder         2636.973333
Gladys Colclough      2560.548148
Hayden Neloms         2543.093458
James Ascencio        3063.207407
Jonathan Berthelot    1666.000000
Kami Bicknell         1818.712644
Kary Hendrixson       2173.674641
Lajuana Vencill       1532.535433
Markita Hansen        2529.169231
Marty Freudenburg     2386.844262
Maureen Marcano       2351.644295
Moses Frase           1606.062016
Niesha Huffines       1685.342857
Reed Clapper          2827.974194
Rosalina Dieter       3269.486111
Rosie Papadopoulos    2950.884615
Versie Hillebrand     1066.437500
Vicki Laflamme        2164.687783
Violet Mclelland      1011.729508
Wilburn Farren        2866.181818
Za

In [189]:
df.groupby("sales_agent")["opportunity_id"].count()

sales_agent
Anna Snelling         448
Boris Faz             210
Cassey Cress          346
Cecily Lampkin        203
Corliss Cosme         310
Daniell Hammack       259
Darcel Schlecht       747
Donn Cantrell         275
Elease Gluck          177
Garret Kinder         123
Gladys Colclough      317
Hayden Neloms         202
James Ascencio        267
Jonathan Berthelot    345
Kami Bicknell         362
Kary Hendrixson       438
Lajuana Vencill       311
Markita Hansen        306
Marty Freudenburg     281
Maureen Marcano       285
Moses Frase           260
Niesha Huffines       239
Reed Clapper          237
Rosalina Dieter       160
Rosie Papadopoulos    160
Versie Hillebrand     361
Vicki Laflamme        451
Violet Mclelland      261
Wilburn Farren        110
Zane Levy             349
Name: opportunity_id, dtype: int64

In [192]:
won_by_agent = df[df["deal_stage"]== "Won"].groupby("sales_agent")["opportunity_id"].count()
lost_by_agent = df[df["deal_stage"]== "Lost"].groupby("sales_agent")["opportunity_id"].count()

print(lost_by_agent)

sales_agent
Anna Snelling         128
Boris Faz              52
Cassey Cress           98
Cecily Lampkin         53
Corliss Cosme          79
Daniell Hammack        73
Darcel Schlecht       204
Donn Cantrell         117
Elease Gluck           46
Garret Kinder          48
Gladys Colclough       97
Hayden Neloms          45
James Ascencio         71
Jonathan Berthelot     93
Kami Bicknell          98
Kary Hendrixson       126
Lajuana Vencill       104
Markita Hansen         97
Marty Freudenburg      72
Maureen Marcano        64
Moses Frase            66
Niesha Huffines        70
Reed Clapper           82
Rosalina Dieter        38
Rosie Papadopoulos     43
Versie Hillebrand      88
Vicki Laflamme        126
Violet Mclelland       71
Wilburn Farren         24
Zane Levy             100
Name: opportunity_id, dtype: int64


In [193]:
closed_by_agent =  won_by_agent + lost_by_agent
print(closed_by_agent)

sales_agent
Anna Snelling         336
Boris Faz             153
Cassey Cress          261
Cecily Lampkin        160
Corliss Cosme         229
Daniell Hammack       187
Darcel Schlecht       553
Donn Cantrell         275
Elease Gluck          126
Garret Kinder         123
Gladys Colclough      232
Hayden Neloms         152
James Ascencio        206
Jonathan Berthelot    264
Kami Bicknell         272
Kary Hendrixson       335
Lajuana Vencill       231
Markita Hansen        227
Marty Freudenburg     194
Maureen Marcano       213
Moses Frase           195
Niesha Huffines       175
Reed Clapper          237
Rosalina Dieter       110
Rosie Papadopoulos    121
Versie Hillebrand     264
Vicki Laflamme        347
Violet Mclelland      193
Wilburn Farren         79
Zane Levy             261
Name: opportunity_id, dtype: int64


In [194]:
win_rate_by_agent = won_by_agent/(closed_by_agent)*100
print(win_rate_by_agent)

sales_agent
Anna Snelling         61.904762
Boris Faz             66.013072
Cassey Cress          62.452107
Cecily Lampkin        66.875000
Corliss Cosme         65.502183
Daniell Hammack       60.962567
Darcel Schlecht       63.110307
Donn Cantrell         57.454545
Elease Gluck          63.492063
Garret Kinder         60.975610
Gladys Colclough      58.189655
Hayden Neloms         70.394737
James Ascencio        65.533981
Jonathan Berthelot    64.772727
Kami Bicknell         63.970588
Kary Hendrixson       62.388060
Lajuana Vencill       54.978355
Markita Hansen        57.268722
Marty Freudenburg     62.886598
Maureen Marcano       69.953052
Moses Frase           66.153846
Niesha Huffines       60.000000
Reed Clapper          65.400844
Rosalina Dieter       65.454545
Rosie Papadopoulos    64.462810
Versie Hillebrand     66.666667
Vicki Laflamme        63.688761
Violet Mclelland      63.212435
Wilburn Farren        69.620253
Zane Levy             61.685824
Name: opportunity_id, dtype:

In [195]:
df[df["deal_stage"]=="Won"].groupby("product")["opportunity_id"].count()

product
GTK 500            15
GTX Basic         915
GTX Plus Basic    653
GTX Plus Pro      479
GTXPro            729
MG Advanced       654
MG Special        793
Name: opportunity_id, dtype: int64

In [196]:
df[df["deal_stage"]=="Lost"].groupby("product")["opportunity_id"].count()

product
GTK 500            10
GTX Basic         521
GTX Plus Basic    398
GTX Plus Pro      266
GTXPro            418
MG Advanced       430
MG Special        430
Name: opportunity_id, dtype: int64

In [197]:
win_rate_by_product = Won_by_product/(Won_by_product+lost_by_product)*100
print(win_rate_by_product)

product
GTK 500           60.000000
GTX Basic         63.718663
GTX Plus Basic    62.131304
GTX Plus Pro      64.295302
GTXPro            63.557105
MG Advanced       60.332103
MG Special        64.840556
Name: opportunity_id, dtype: float64


In [202]:
df[df["deal_stage"]=="Won"].groupby("product")["close_value"].sum()

product
GTK 500            400612.0
GTX Basic          499263.0
GTX Plus Basic     705275.0
GTX Plus Pro      2629651.0
GTXPro            3510578.0
MG Advanced       2216387.0
MG Special          43768.0
Name: close_value, dtype: float64

In [203]:
df[df["deal_stage"]=="Won"].groupby("product")["close_value"].mean()

product
GTK 500           26707.466667
GTX Basic           545.642623
GTX Plus Basic     1080.053599
GTX Plus Pro       5489.876827
GTXPro             4815.607682
MG Advanced        3388.970948
MG Special           55.192938
Name: close_value, dtype: float64